<a href="https://colab.research.google.com/github/Sank3t-Pand3y/MyPractiseML/blob/main/optuna_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 5.3 MB/s eta 0:00:00


In [2]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [3]:
# Load the Pima Indian Diabetes from sklearn
# Note: Scikit-learn's builti-in 'load_diabetes' is a regression datasets.
# We will load the actual diabetes dataset from an external source
import pandas as pd




In [4]:
# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

In [5]:
# load the dataset
df = pd.read_csv(url, names = columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [6]:
import numpy as np



In [11]:
# Replace zero values with NaN in columns where zero is not a valid value
# That means :- We are Replacing 0 (zero) with NaN

cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)


# Impute the missing values with the mean of the respective column
# That means :- We are Replacing NaN with the mean of each particular column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [12]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.3, random_state=42)


# Optional : Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training data shape: {X_train.shape}')
print(f'Testing data shape: {X_test.shape}')

Training data shape: (537, 8)
Testing data shape: (231, 8)


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
     # n_estimators means number of the decision tree
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [16]:
# Create a study object and optimize the objective function
# direction='maximize' that is maximized cause we need the Maximum Value

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters


[I 2025-11-23 07:06:44,000] A new study created in memory with name: no-name-4e57c9fb-5e80-4013-b11f-33679317024b
[I 2025-11-23 07:06:46,348] Trial 0 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 183, 'max_depth': 11}. Best is trial 0 with value: 0.7672253258845437.
[I 2025-11-23 07:06:49,145] Trial 1 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 200, 'max_depth': 20}. Best is trial 1 with value: 0.7728119180633147.
[I 2025-11-23 07:06:52,465] Trial 2 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 198, 'max_depth': 19}. Best is trial 2 with value: 0.7746741154562384.
[I 2025-11-23 07:06:53,698] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 60, 'max_depth': 12}. Best is trial 2 with value: 0.7746741154562384.
[I 2025-11-23 07:06:55,197] Trial 4 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 114, 'max_depth': 5}. Best is trial 2 with value: 0.774674

In [17]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7783985102420857
Best hyperparameters: {'n_estimators': 177, 'max_depth': 16}


# A Complete New MODEL with Best Result from Trials of Hyperparamters

In [18]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


# There are also other Samplers in Optuna

In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [22]:
# We use RandomSampler OPTUNA

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2025-11-23 07:18:43,564] A new study created in memory with name: no-name-c11d194e-dae5-41c3-91ef-67805059f204
[I 2025-11-23 07:18:44,643] Trial 0 finished with value: 0.7765363128491621 and parameters: {'n_estimators': 60, 'max_depth': 8}. Best is trial 0 with value: 0.7765363128491621.
[I 2025-11-23 07:18:46,571] Trial 1 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 138, 'max_depth': 16}. Best is trial 0 with value: 0.7765363128491621.
[I 2025-11-23 07:18:48,458] Trial 2 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 190, 'max_depth': 8}. Best is trial 0 with value: 0.7765363128491621.
[I 2025-11-23 07:18:49,604] Trial 3 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 125, 'max_depth': 3}. Best is trial 0 with value: 0.7765363128491621.
[I 2025-11-23 07:18:50,336] Trial 4 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 69, 'max_depth': 15}. Best is trial 0 with value: 0.776536312

In [23]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7821229050279329
Best hyperparameters: {'n_estimators': 118, 'max_depth': 7}


In [25]:
# A Complete New MODEL with Best Result from Trials of Hyperparamters


from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.75


# Another Type of Sampler is Grid Search.

# Here, in Grid Search, We Define the search_space explicity i.e Outside of Objective Function.

In [26]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [27]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2025-11-23 07:23:44,726] A new study created in memory with name: no-name-34bb7c17-3416-46c7-9451-47d63126b0c6
[I 2025-11-23 07:23:46,610] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-11-23 07:23:48,604] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2025-11-23 07:23:49,129] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2025-11-23 07:23:50,190] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2025-11-23 07:23:51,233] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [28]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [29]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


# OPTUNA Visualizations

In [30]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [32]:
# 1. Optimization History
plot_optimization_history(study).show()

# Objective Values = Accuracy Value

In [33]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()


# This Plot helps you to understand between Accuracy, max_depth and n_estimators

In [34]:
# 3. Slice Plot
# In Slice Plot, We indivisually plot each hyperparameters with Objective Value i.e Accuracy
plot_slice(study).show()

In [35]:
# 4. Contour Plot
plot_contour(study).show()

In [37]:
# 5. Hyperparameter Importance
# This Graph Shows which hyperparameter is the most important. Currently, it is between max_depth and n_estimators
plot_param_importances(study).show()

# The Best Feature of the OPTUNA is that We can find the best Algorithm for the Dataset with the best Hyperparameters.

# That Means We can Optimize the Multiple ML models

In [38]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [39]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [40]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2025-11-23 07:54:20,728] A new study created in memory with name: no-name-a0a428cf-77d5-456d-9b95-cdbf072fd55c
[I 2025-11-23 07:54:20,797] Trial 0 finished with value: 0.756052141527002 and parameters: {'classifier': 'SVM', 'C': 10.600508182878743, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 0 with value: 0.756052141527002.
[I 2025-11-23 07:54:26,903] Trial 1 finished with value: 0.750465549348231 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 245, 'learning_rate': 0.01778683339377578, 'max_depth': 10, 'min_samples_split': 10, 'min_samples_leaf': 10}. Best is trial 0 with value: 0.756052141527002.
[I 2025-11-23 07:54:30,635] Trial 2 finished with value: 0.7355679702048418 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 126, 'learning_rate': 0.033689026943473636, 'max_depth': 17, 'min_samples_split': 3, 'min_samples_leaf': 4}. Best is trial 0 with value: 0.756052141527002.
[I 2025-11-23 07:54:30,698] Trial 3 finished with value: 0.707635009

In [41]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.12252172302055128, 'kernel': 'linear', 'gamma': 'scale'}
Best trial accuracy: 0.7895716945996275


In [42]:
# We can Print all the Trials that Occuried inside the DataFrame
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.756052,2025-11-23 07:54:20.734698,2025-11-23 07:54:20.797299,0 days 00:00:00.062601,10.600508,NaN,SVM,auto,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.750466,2025-11-23 07:54:20.798525,2025-11-23 07:54:26.903606,0 days 00:00:06.105081,NaN,NaN,GradientBoosting,NaN,NaN,0.017787,10.0,10.0,10.0,245.0,COMPLETE
2,2,0.735568,2025-11-23 07:54:26.905656,2025-11-23 07:54:30.635138,0 days 00:00:03.729482,NaN,NaN,GradientBoosting,NaN,NaN,0.033689,17.0,4.0,3.0,126.0,COMPLETE
3,3,0.707635,2025-11-23 07:54:30.636079,2025-11-23 07:54:30.697982,0 days 00:00:00.061903,29.711705,NaN,SVM,auto,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
4,4,0.772812,2025-11-23 07:54:30.698901,2025-11-23 07:54:31.492038,0 days 00:00:00.793137,NaN,False,RandomForest,NaN,NaN,NaN,6.0,6.0,3.0,165.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.787709,2025-11-23 07:55:07.291188,2025-11-23 07:55:07.325793,0 days 00:00:00.034605,0.178969,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.783985,2025-11-23 07:55:07.326664,2025-11-23 07:55:07.362320,0 days 00:00:00.035656,0.405581,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.765363,2025-11-23 07:55:07.363181,2025-11-23 07:55:10.441003,0 days 00:00:03.077822,NaN,NaN,GradientBoosting,NaN,NaN,0.010790,18.0,9.0,3.0,251.0,COMPLETE
98,98,0.789572,2025-11-23 07:55:10.441961,2025-11-23 07:55:10.475335,0 days 00:00:00.033374,0.135683,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [44]:
# Each Algorithm kati choti RUN vayo.
# i.e. 100 trails thiyo
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,69
RandomForest,21
GradientBoosting,10


In [46]:
# Each Algorithm ko Average kati cha, we can find it
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.745438
RandomForest,0.769354
SVM,0.775754
